# SC3000 DPO — reproducible final E5 training notebook

**Final selected model:** Round9 Random E5 (historical development Val768 614/768, Teacher680 521/680). This is a multi-stage curriculum, **not** five epochs directly from SFT.

This notebook provides the **executable training loop**, explicit data preparation, hash-based provenance, epoch-boundary resume, identical course loss and greedy inference, and a final dpo/dpo.pt export. Open from repository root or dpo/, set FAST_SMOKE=False, and choose **Kernel → Restart & Run All**.

**Historical chain (all stages start fresh AdamW; see stage-specific cosine schedules below):**

1. SFT → Round3 full 100k structured, five epochs, LR 1e-4.
2. Round3 → Round4 Treatment **epoch 1 of a two-epoch cosine schedule**, LR 2e-5.
3. Round4 E1 → Round5 Original (one-step because), one epoch, LR 1e-5.
4. Round5 Original → Round7 Replay, one epoch, LR 1e-5.
5. Round7 Replay → Round9 Random E1, one epoch, LR 1e-5.
6. Random E1 → Random E2, one epoch, LR 1e-5, **fresh optimizer** (original E1 did not store optimizer state).
7. Random E2 → Random E3, E4, **E5** (three actual epochs on a **four-epoch cosine schedule**), LR 1e-5, one continuous optimizer across E3–E5.

**Required starter files:** supplied sft/gpt.pt, sft/meta.pkl, the course model.py, and source scripts from this GitHub repository. The notebook automatically builds missing source datasets using the audited generator scripts (their SHA256 is included in its report). Do **not** upload only dpo.pt and claim reproduction from scratch. Preserve all generated training JSON and the generator scripts with the assignment package; the historical project datasets are large and not tracked by Git.

**Integrity:** Each dataset and parent checkpoint is hashed, factual answers/format are checked, stage checkpoints store optimizer, scheduler, Python/Torch RNG and shuffled training order. If interrupted, rerun **Run All**; completed stages are verified and skipped. Historical checkpoints elsewhere in the repo are never overwritten. Round3 final Test768 is **never evaluated** for model selection; frozen Val768 and Seen-heavy Teacher680 are development benchmarks.

**Limits:** Exact numerical identity with the archived historical E5 cannot be guaranteed across GPUs, PyTorch releases, nondeterministic CUDA operations or different source code commits. This notebook reports a comparison when the archived E5 is available rather than falsely promising byte-level parity. Historical archived scores are **reference results**, not a claim that the new run has already been executed. The TC1 Slurm 6-hour per-job QoS may require **rerunning Run All to resume**; a single 6-hour allocation cannot be guaranteed to complete the full historical chain.

In [ ]:
# STEP 1 — Imports, deterministic protocol and runtime paths
from __future__ import annotations
import os, sys, json, pickle, random, math, hashlib, shutil, time, re, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import torch
import torch.nn.functional as F

workdir = Path.cwd().resolve()
if (workdir / "model.py").is_file() and (workdir / "dpo" / "dpo.ipynb").exists():
    repo_root = workdir
elif (workdir / "dpo.ipynb").is_file() and (workdir.parent / "model.py").is_file():
    repo_root = workdir.parent
else:
    raise RuntimeError("Open from NanoGPT-Math root/ or NanoGPT-Math/dpo/.")

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "scripts"))
from model import GPT, GPTConfig

# Keep the originally supplied pretrained 8.84M-parameter NanoGPT checkpoint.
SFT = repo_root / "sft" / "gpt.pt"
META = repo_root / "sft" / "meta.pkl"
for required in (SFT, META):
    if not required.is_file():
        raise FileNotFoundError(f"Missing REQUIRED course-provided input: {required}")

with META.open("rb") as fh:
    vocab = pickle.load(fh)
stoi, itos = vocab["stoi"], vocab["itos"]
assert len(stoi) == len(itos) == 74, "Expected original 74-character vocabulary"
assert stoi["\n"] == 0 and stoi["."] == 7, "Original stopping IDs changed"

# FULL: 13 historical 100k training epochs across seven stages.
# SMOKE: replace each 100k stage by a 128-row mini-stage and NEVER publish dpo.pt.
FAST_SMOKE = False
AUTO_BUILD_MISSING_DATA = True
MAX_NEW_TOKENS = 80
BATCH_SIZE = 64
SEED = 42
BETA = 0.5
TRAIN_BASE_LENGTH = 64
device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_DIR = (repo_root / "dpo" / "experiments" /
           ("final_notebook_smoke" if FAST_SMOKE else "final_notebook_e5_reproduction"))
RUN_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for part in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def save_json_once(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        raise FileExistsError(f"Will not overwrite frozen artifact: {path}")
    temporary = path.with_suffix(path.suffix + ".partial")
    if temporary.exists():
        raise RuntimeError(f"Stale incomplete artifact, inspect first: {temporary}")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n",
                         encoding="utf-8")
    temporary.rename(path)

def record_environment():
    try:
        git_rev = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo_root,
                                text=True, check=True, capture_output=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        git_rev = "unknown (archive export)"
    return {"utc":datetime.now(timezone.utc).isoformat(),
            "python":sys.version, "torch":torch.__version__,
            "cuda_runtime":torch.version.cuda,
            "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "git_revision":git_rev,
            "sft_sha256":sha256(SFT), "tokenizer_meta_sha256":sha256(META)}

env_record=record_environment()
print(json.dumps(env_record, indent=2), flush=True)
if FAST_SMOKE:
    print("SMOKE mode: never export its model as formal dpo.pt")
elif device != "cuda":
    raise RuntimeError("Full 13-epoch reproduction requires CUDA. "
                       "Use a Slurm GPU/Jupyter allocation or set FAST_SMOKE=True.")

## Task 1 — Preference data construction and isolation

The data builders are the original audited project implementation. This cell calls them **automatically**, in source-dependency order, if their immutable manifest is absent; it never silently regenerates a frozen dataset. Each builder verifies exact arithmetic, sentence format, 74-character vocabulary, 256-character NanoGPT block limit, and canonical-math-fact splits (seed 42).

Files in the historical path:

- Round1A → Round1B integer-only → Round1C historical fixed OOD256 (used only to isolate later generated facts; **not** an optimization objective here).
- Round3: 100k preference pairs; validation 768; **final test is sealed**.
- Round4 Treatment: 70k common replay + 15k hard seen training + 15k new train facts; extended validation kept out of training.
- Round5 Original: Round4 Treatment questions and one-step explanations preserved; compare with the alternative decomposition branch separately in the research log.
- Round7 Replay: controlled same-size 100k replay.
- Round9 Random: 70k common old + 30k additional training-partition questions, frozen after Round7 checkpoint exists.

Training format: positive = question + ' The answer is [integer] because [verified equality].'; negative = same question + " Sorry, I don't know.". We keep historical repeated exposures and log unique prompt/fact counts. Changing them here would constitute a new experiment, not the same E5.

In [ ]:
# STEP 2 — Generate *missing* immutable data automatically (no manual shell steps).
# All builder programs live in this same GitHub repository, and their
# exact source SHA256 is stored in the final run provenance.
GENERATOR_PLAN = [
    ("Round1A frozen source", repo_root/"dpo/data/round1_full/round1_manifest.json",
     "scripts/generate_round1.py", ["--profile", "full"]),
    ("Round1B integer source", repo_root/"dpo/experiments/round1b_integer/round1b_manifest.json",
     "scripts/generate_round1b.py", []),
    ("Round1C fixed validation", repo_root/"dpo/experiments/round1c_incorrect/round1c_manifest.json",
     "scripts/generate_round1c.py", []),
    ("Round3 structured 100k", repo_root/"dpo/data/round3_structured_full/manifest.json",
     "scripts/generate_round3_structured.py", ["--profile", "full"]),
    ("Round4 matched data", repo_root/"dpo/data/round4_matched_full/manifest.json",
     "scripts/build_round4_data.py", ["--profile", "full"]),
    ("Round5 one-step data", repo_root/"dpo/data/round5_decomposition/manifest.json",
     "scripts/build_round5_decomposition.py", []),
    ("Round7 paired replay", repo_root/"dpo/data/round7_targeted_full/manifest.json",
     "scripts/build_round7_targeted_data.py", ["--profile", "full"]),
]
generator_provenance = {}

for name, manifest_path, script_rel, args in GENERATOR_PLAN:
    script = repo_root / script_rel
    if not script.is_file():
        raise FileNotFoundError(f"Reproducibility requires source generator {script_rel}")
    generator_provenance[script_rel] = sha256(script)
    if not manifest_path.exists():
        if not AUTO_BUILD_MISSING_DATA:
            raise FileNotFoundError(f"{name}: missing frozen source {manifest_path}")
        print(f"GENERATE {name}: {script_rel} {args}", flush=True)
        subprocess.run([sys.executable, str(script), *args],
                       cwd=repo_root, check=True)
    if not manifest_path.is_file():
        raise RuntimeError(f"{name} generation did not produce {manifest_path}")
    generator_provenance[str(manifest_path.relative_to(repo_root))] = sha256(manifest_path)
    print(f"PREPARED {name}: {manifest_path.relative_to(repo_root)}", flush=True)

# Frozen teacher-style 680 is for retention diagnostics only; its
# arithmetic facts were historically seen, never to be called unseen OOD.
TEACHER_DIRS = [
    (repo_root/"dpo/benchmarks/teacher_style_400_v1", 400),
    (repo_root/"dpo/benchmarks/teacher_style_extension_280_v1", 280),
]
for directory, expected_n in TEACHER_DIRS:
    questions = read_json(directory/"questions.json")
    manifest = read_json(directory/"manifest.json")
    assert len(questions) == expected_n
    assert sha256(directory/"questions.json") == manifest["questions_sha256"]
    generator_provenance[str((directory/"questions.json").relative_to(repo_root))] = sha256(directory/"questions.json")
print("PASS: 400 + 280 frozen teacher questions", flush=True)

## Task 2 — Model and original course preference objective

We intentionally preserve the instructor's original **whole-sequence mean log-probability** loss (not standard frozen-reference DPO). The positive/negative are teacher-forced character sequences: the GPT does **not** generate either answer during training.

For each pair, sequence log-probability is the negative average token cross entropy over non-padding tokens, including the math question. With instructor beta = 0.5:

Loss = mean[softplus(-(logp_chosen - logp_rejected)/beta)] - 0.1 * mean(logp_chosen).

The final term encourages the correct answer sequence itself. All training stages use the original GPTConfig, 74-character tokenizer, block_size=256, **same four appended newline characters**, and max_length adjusted upward if necessary to avoid truncating valid examples.

In [ ]:
# STEP 3 — Course-faithful teacher-forcing loss and preprocessing.
from generate_round1 import parse_relation, stable_split

def encode(s):
    return [stoi[ch] for ch in s]

def pad_or_truncate(chars, max_length):
    return chars[-max_length:] if len(chars)>max_length else chars+[0]*(max_length-len(chars))

def compute_logprob(gpt, batch_ids):
    inputs = batch_ids[:, :-1]
    targets = batch_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)),
                           targets.reshape(-1), ignore_index=0,
                           reduction="none").reshape_as(targets)
    mask = (targets!=0).float()
    return -(loss*mask).sum(dim=1)/mask.sum(dim=1)

def course_preference_loss(gpt, negatives, positives):
    neg_lp = compute_logprob(gpt, negatives)
    pos_lp = compute_logprob(gpt, positives)
    return -F.logsigmoid((pos_lp-neg_lp)/BETA).mean()-0.1*pos_lp.mean()

def get_batches(lines, batch_size, max_length, target_device):
    # EXACT instructor rule: shuffle rows in place, discard trailing <batch.
    random.shuffle(lines)
    for i in range(0,len(lines),batch_size):
        batch=lines[i:i+batch_size]
        if len(batch)<batch_size:continue
        neg=[pad_or_truncate(encode(x["negative"]+"\n\n\n\n"),max_length) for x in batch]
        pos=[pad_or_truncate(encode(x["positive"]+"\n\n\n\n"),max_length) for x in batch]
        yield (torch.tensor(neg,dtype=torch.long,device=target_device),
               torch.tensor(pos,dtype=torch.long,device=target_device))

def verify_pairs(path, expected_count=100000, smoke=False):
    source = read_json(path)
    if len(source)!=expected_count:
        raise RuntimeError(f"{path}: expected {expected_count}, found {len(source)}")
    rows = source[:128] if smoke else source
    max_required = TRAIN_BASE_LENGTH
    unique_prompts=set()
    unique_facts=set()
    for i,row in enumerate(rows):
        pos,neg=row["positive"],row["negative"]
        marker=" The answer is "
        if marker not in pos or " because " not in pos:
            raise ValueError(f"{path.name} row {i}: incorrect chosen format")
        question,answer_suffix=pos.split(marker,1)
        answer=answer_suffix.split(" because ",1)[0]
        if not re.fullmatch(r"[+-]?\d+",answer) or not pos.endswith("."):
            raise ValueError(f"{path.name} row {i}: malformed integer chosen")
        if neg!=question+" Sorry, I don't know.":
            raise ValueError(f"{path.name} row {i}: changed rejection/prompt")
        if set(pos+neg)-set(stoi):
            raise ValueError(f"{path.name} row {i}: tokenizer OOV")
        max_required=max(max_required,len(pos)+4,len(neg)+4)
        unique_prompts.add(question)
        unique_facts.add(parse_relation(question,answer))
    # Original NanoGPT block size=256; 64 was only the initial pack length.
    if max_required>256:
        raise ValueError(f"{path.name}: {max_required} exceeds model block_size")
    info={"source":str(path.relative_to(repo_root)),"sha256":sha256(path),
          "rows":len(rows),"unique_prompts":len(unique_prompts),
          "unique_facts":len(unique_facts),"max_length":max_required}
    return rows,info

def model_from_checkpoint(path):
    obj=torch.load(path,map_location="cpu",weights_only=True)
    args=obj["model_args"]
    model=GPT(GPTConfig(**args))
    state=obj.get("model_state_dict",obj.get("model"))
    if state is None:raise RuntimeError(f"{path}: missing state dict")
    model.load_state_dict({k.removeprefix("_orig_mod."):v for k,v in state.items()},strict=True)
    return model,args

# Sanity checks: answer-first comparison, no generated completion needed in training.
assert encode("3*17=?")
assert abs(float(F.logsigmoid(torch.tensor(0.0)))+math.log(2))<1e-6
print("PASS: course loss, tokenizer, model loading and batch conventions",flush=True)

## Task 3 — Faithful staged checkpoint plan

These are **historical stage boundaries**, not experiments optimized after seeing the final test. Each stage initializes a fresh optimizer with seed 42 and trains on exactly 100,000 pair rows (1,562 complete minibatches/epoch, batch 64). The two deliberate partial-schedule checkpoints are crucial:

- Round4 treatment is checkpointed at epoch 1, while its cosine scheduler is configured for 2 epochs.
- The selected E5 is saved at the third epoch of a **four-epoch** cosine schedule beginning with E3. Training only a three-epoch cosine schedule would not reproduce the historical checkpoint.
- Random E1 and E2 were separate 1-epoch runs with fresh AdamW, followed by a separate E3–E6 multi-epoch run. E2→E3 is another optimizer restart.

All stage training, teacher forcing, batch order, cosine scheduler, stop conditions, checkpoint persistence, and resume logic are **implemented in the notebook** below. No train_round*.py module is invoked during optimization. Original generator scripts are invoked only for reproducible **data construction**.

In [ ]:
# STEP 4 — Historical stage definitions; data SHA must agree with its frozen manifest.
R3 = repo_root / "dpo/data/round3_structured_full"
R4 = repo_root / "dpo/data/round4_matched_full"
R5 = repo_root / "dpo/data/round5_decomposition"
R7 = repo_root / "dpo/data/round7_targeted_full"
R9 = repo_root / "dpo/data/round9_balanced_full"

STAGE1_TO_4=[
    {"id":"round3_structured", "data":R3/"pos_neg_pairs.json",
     "manifest":R3/"manifest.json","hash_section":"sha256","hash_key":"train",
     "lr":1e-4,"actual_epochs":5,"schedule_epochs":5},
    {"id":"round4_treatment_e1","data":R4/"treatment_pairs.json",
     "manifest":R4/"manifest.json","hash_section":"file_sha256","hash_key":"treatment",
     "lr":2e-5,"actual_epochs":1,"schedule_epochs":2},
    {"id":"round5_original","data":R5/"original_pairs.json",
     "manifest":R5/"manifest.json","hash_section":"file_sha256","hash_key":"original",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round7_replay","data":R7/"replay_pairs.json",
     "manifest":R7/"manifest.json","hash_section":"file_sha256","hash_key":"replay",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
]
EXPECTED_RANDOM_SHA="30519fe6883f175e67791a0461afe23e9212f38d974778c4bd5f552d3e26e3fa"

def stage_protocol(spec, parent_path, data_info):
    return {"stage":spec["id"],"source_data":data_info["source"],
            "data_sha256":data_info["sha256"],"parent_sha256":sha256(parent_path),
            "batch_size":16 if FAST_SMOKE else BATCH_SIZE,
            "lr":spec["lr"],"actual_epochs":1 if FAST_SMOKE else spec["actual_epochs"],
            "schedule_epochs":spec["schedule_epochs"],
            "seed":SEED,"beta":BETA,"steps_per_epoch":len(data_info_rows)//(16 if FAST_SMOKE else BATCH_SIZE)
            if False else None,
            "max_length":data_info["max_length"],"objective":"course-full-sequence-mean",
            "optimizer":"AdamW(weight_decay=0.01)", "scheduler":"CosineAnnealingLR(eta_min=0.1*lr)",
            "fast_smoke":FAST_SMOKE}

# Basic contract checks (also repeated in the resumable stage runner).
assert [x["id"] for x in STAGE1_TO_4]==[
    "round3_structured","round4_treatment_e1","round5_original","round7_replay"]
assert [x["actual_epochs"] for x in STAGE1_TO_4]==[5,1,1,1]
assert [x["schedule_epochs"] for x in STAGE1_TO_4]==[5,2,1,1]
print("PASS: 4 historical stages and their original cosine schedules")

## Task 3 — Exact training loop and epoch-boundary resume

For every epoch, call the instructor's get_batches semantics (shuffle in place, pad with four newline IDs, drop incomplete final batch). Compute two log-probabilities under teacher forcing, apply the original Course Preference Loss, clip gradient norm to 1, then advance AdamW and cosine scheduler **once per optimizer step**.

Each completed epoch saves model weights, optimizer, scheduler, Python RNG, PyTorch CPU/CUDA RNG, shuffled order, the parent SHA and dataset SHA. A second Run All verifies those files and continues automatically; an incomplete/stale checkpoint does not get silently overwritten. Only the final verified E5 is eligible for exporting to dpo/dpo.pt.

In [ ]:
# STEP 5 — Training engine, written inline for Notebook "Run All".
def atomic_save_torch(target, obj):
    target=Path(target)
    if target.exists():
        raise FileExistsError(f"Refusing to overwrite checkpoint: {target}")
    tmp=target.with_suffix(target.suffix+".partial")
    if tmp.exists():
        raise RuntimeError(f"Incomplete temporary checkpoint: {tmp}. Inspect first.")
    torch.save(obj, tmp)
    tmp.rename(target)

def stage_paths(stage_id):
    folder=RUN_DIR/stage_id
    return folder,folder/"progress.json"

def stage_finished_checkpoint(spec):
    folder,_=stage_paths(spec["id"])
    n=1 if FAST_SMOKE else spec["actual_epochs"]
    return folder/f"epoch_{n:02d}.pt"

def train_stage(spec, parent_path):
    assert Path(parent_path).is_file(),f"Missing parent checkpoint {parent_path}"
    data_file=Path(spec["data"])
    source_m=read_json(spec["manifest"])
    expected=source_m[spec["hash_section"]][spec["hash_key"]]
    if sha256(data_file)!=expected:
        raise RuntimeError(f"Data SHA256 does not match frozen source manifest: {data_file}")
    allrows,info=verify_pairs(data_file,smoke=FAST_SMOKE)
    batch=16 if FAST_SMOKE else BATCH_SIZE
    actual=1 if FAST_SMOKE else spec["actual_epochs"]
    schedule_epochs=spec["schedule_epochs"]
    steps=len(allrows)//batch
    if steps<=0:raise RuntimeError("Not enough preferences for one minibatch")
    protocol={
        "stage":spec["id"],"parent_sha256":sha256(parent_path),
        "dataset_sha256":info["sha256"],
        "source_manifest_sha256":sha256(spec["manifest"]),
        "seed":SEED,"batch_size":batch,"steps_per_epoch":steps,
        "actual_epochs":actual,"cosine_schedule_epochs":schedule_epochs,
        "learning_rate":spec["lr"],"loss":"instructor full-sequence mean",
        "max_length":info["max_length"],"beta":BETA,
        "model_block_size":256,"fast_smoke":FAST_SMOKE
    }
    stage_dir,progress_path=stage_paths(spec["id"])
    if progress_path.is_file():
        progress=read_json(progress_path)
        if progress["protocol"]!=protocol:
            raise RuntimeError("Refusing incompatible resume: stage data/schedule/parent changed")
        finished=progress["completed_epochs"]
        if not 1<=finished<=actual:
            raise RuntimeError("Invalid saved epoch offset")
        epoch_file=stage_dir/f"epoch_{finished:02d}.pt"
        if sha256(epoch_file)!=progress["last_checkpoint_sha256"]:
            raise RuntimeError(f"Corrupt stage checkpoint {epoch_file}")
        if finished==actual:
            print(f"SKIP verified {spec['id']}: {finished}/{actual} complete",flush=True)
            return epoch_file
        print(f"RESUME {spec['id']} from epoch {finished}/{actual}",flush=True)
    elif stage_dir.exists():
        raise RuntimeError(f"Stale/incomplete stage folder: {stage_dir}; "
                           "inspect or rename it instead of deleting hidden work")
    else:
        stage_dir.mkdir(parents=True,exist_ok=False)
        save_json_once(stage_dir/"protocol.json",protocol)
        finished=0
        progress=None

    # Previous Python/CUDA seed reset BEFORE constructing GPT, just like all
    # historical train_round*.py stage entry points.
    random.seed(SEED)
    torch.manual_seed(SEED)
    if device=="cuda":torch.cuda.manual_seed_all(SEED)
    gpt, model_args=model_from_checkpoint(parent_path)
    if model_args["vocab_size"]!=len(stoi) or model_args["block_size"]!=256:
        raise RuntimeError("Original character tokenizer / model architecture changed")
    gpt.to(device).train()
    opt=torch.optim.AdamW(gpt.parameters(),lr=spec["lr"],weight_decay=0.01)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,T_max=schedule_epochs*steps,eta_min=spec["lr"]*0.1)
    lines=allrows

    if finished:
        ck=torch.load(epoch_file,map_location="cpu",weights_only=True)
        if ck["stage_protocol"]!=protocol or ck["epoch"]!=finished:
            raise RuntimeError("Resume checkpoint metadata mismatch")
        gpt.load_state_dict(ck["model_state_dict"],strict=True)
        opt.load_state_dict(ck["optimizer_state_dict"])
        scheduler.load_state_dict(ck["scheduler_state_dict"])
        lines=ck["shuffled_lines"]
        random.setstate(ck["python_rng_state"])
        torch.set_rng_state(ck["torch_rng_state"])
        if device=="cuda":
            torch.cuda.set_rng_state_all(ck["cuda_rng_states"])
        if len(lines)!=len(allrows):
            raise RuntimeError("Saved shuffle order lost rows")

    for epoch in range(finished+1,actual+1):
        gpt.train()
        accumulated=0.
        updated=0
        for step,(neg,pos) in enumerate(get_batches(lines,batch,info["max_length"],device),1):
            opt.zero_grad(set_to_none=True)
            loss=course_preference_loss(gpt,neg,pos)
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError(f"Non-finite loss: {spec['id']} epoch={epoch} step={step}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gpt.parameters(),max_norm=1.0)
            opt.step()
            scheduler.step()
            updated+=1
            accumulated+=float(loss.item())
            if step==1 or step%250==0 or step==steps:
                print(f"TRAIN {spec['id']} epoch={epoch}/{actual} "
                      f"step={step}/{steps} loss={loss.item():.5f} "
                      f"lr={scheduler.get_last_lr()[0]:.9g}",flush=True)
        if updated!=steps:raise RuntimeError("Optimizer step count mismatch")
        ckpt=stage_dir/f"epoch_{epoch:02d}.pt"
        obj={
            "epoch":epoch,"stage_protocol":protocol,
            "model_state_dict":gpt.state_dict(),"model_args":model_args,
            "optimizer_state_dict":opt.state_dict(),
            "scheduler_state_dict":scheduler.state_dict(),
            "shuffled_lines":lines,"python_rng_state":random.getstate(),
            "torch_rng_state":torch.get_rng_state(),
            "cuda_rng_states":torch.cuda.get_rng_state_all() if device=="cuda" else []
        }
        atomic_save_torch(ckpt,obj)
        history=progress["history"] if progress else []
        history.append({"epoch":epoch,"mean_loss":accumulated/steps,
                        "checkpoint":ckpt.name,"sha256":sha256(ckpt)})
        progress={"protocol":protocol,"completed_epochs":epoch,
                  "last_checkpoint_sha256":sha256(ckpt),"history":history}
        temp=progress_path.with_suffix(".json.partial")
        if temp.exists():raise RuntimeError(f"Stale progress marker: {temp}")
        temp.write_text(json.dumps(progress,indent=2)+"\n",encoding="utf-8")
        temp.replace(progress_path)
        print(f"EPOCH SAVED {spec['id']} {epoch}/{actual} "
              f"mean_loss={accumulated/steps:.6f}",flush=True)
    print(f"STAGE PASS {spec['id']} -> {ckpt.name}",flush=True)
    return ckpt

# Verify the original SFT config without changing any model weights.
sft_obj=torch.load(SFT,map_location="cpu",weights_only=True)
sft_args=sft_obj["model_args"]
assert (sft_args["n_layer"],sft_args["n_head"],sft_args["n_embd"],
        sft_args["vocab_size"],sft_args["block_size"])==(6,6,348,74,256)
del sft_obj
print("PASS: full-stage optimizer, scheduler and checkpoint/resume definitions",flush=True)

## Task 3 — Run the first four historical stages

This cell trains or resumes Round3, Round4 Treatment E1, Round5 Original and Round7 Replay from the original provided SFT weights, in exactly that sequence. No evaluation on a held-out test is performed inside these training stages. Once Round7 Replay exists, the next cell creates/validates Round9 Random data and continues E1→E5.

In [ ]:
# STEP 6 — Sequential, resumable SFT -> Round7 Replay.
stage_outputs={"sft":SFT}
parent=SFT
for spec in STAGE1_TO_4:
    parent=train_stage(spec,parent)
    stage_outputs[spec["id"]]=parent
    print(f"CHAIN {spec['id']}: SHA256 {sha256(parent)[:20]}",flush=True)

ROUND7_PARENT=stage_outputs["round7_replay"]
print(f"PASS: Round7 Replay predecessor created/verified: {ROUND7_PARENT}",flush=True)

## Task 3 — Round9 Random data reproducibility

**Do not regenerate an already frozen Random JSON.** On TC1 the source Random training dataset has SHA256 30519fe6883f175e67791a0461afe23e9212f38d974778c4bd5f552d3e26e3fa. If it exists, assert its hash and reuse the exact 100k pair order.

If it does not exist in a fresh checkout, the notebook invokes the same Round9 deterministic generator after producing the Round7 parent, with all Val/OOD/Extended/Teacher protections. It stores generated data under the **notebook's isolated run folder**, never overwrites historical Round9 outputs, and requires the generated pair file to have the historical SHA256. The Round9 manifest may include a different checkpoint hash if earlier floating point training was not bitwise identical; that is recorded rather than concealed.

In [ ]:
# STEP 7 — Select exact Round9 Random 100k or generate it deterministically.
EXISTING_R9_DATA=R9/"random_pairs.json"
if EXISTING_R9_DATA.exists():
    r9_manifest=read_json(R9/"manifest.json")
    if r9_manifest["version"]!="round9_structured_v1":
        raise RuntimeError("Wrong historical Round9 manifest")
    if sha256(EXISTING_R9_DATA)!=r9_manifest["files_sha256"]["random"]:
        raise RuntimeError("Historical Random100k data SHA256 mismatch")
    random_data_path=EXISTING_R9_DATA
    random_manifest_path=R9/"manifest.json"
else:
    if not AUTO_BUILD_MISSING_DATA:
        raise FileNotFoundError("Missing Round9 Random preference dataset")
    import build_round9_data as b9
    import evaluate_teacher_style_680 as teacher680_module
    # Strictly validate both existing frozen teacher benchmarks.
    teacher680_module.validate()
    generated=RUN_DIR/"notebook_generated_round9"
    if generated.exists():
        if not (generated/"random_pairs.json").exists():
            raise RuntimeError("Incomplete generated Round9 folder; inspect it")
        random_data_path=generated/"random_pairs.json"
        random_manifest_path=generated/"manifest.json"
    else:
        # The original builder expects the parent at a historical absolute
        # path. Use our new Round7 checkpoint, without copying it there.
        historical_base=b9.BASE
        try:
            b9.BASE=ROUND7_PARENT
            r9_pairs,r9_metadata,r9_manifest=b9.make()
        finally:
            b9.BASE=historical_base
        generated.mkdir(parents=True,exist_ok=False)
        for arm in ("replay","random","structured"):
            (generated/f"{arm}_pairs.json").write_bytes(b9.bytes_json(r9_pairs[arm]))
        (generated/"structured_metadata.json").write_bytes(
            b9.bytes_json(r9_metadata))
        (generated/"manifest.json").write_bytes(b9.bytes_json(r9_manifest))
        random_data_path=generated/"random_pairs.json"
        random_manifest_path=generated/"manifest.json"

r9_manifest=read_json(random_manifest_path)
if sha256(random_data_path)!=r9_manifest["files_sha256"]["random"]:
    raise RuntimeError("Generated Random source mismatch with its own manifest")
if sha256(random_data_path)!=EXPECTED_RANDOM_SHA:
    raise RuntimeError(
        "Round9 Random data do NOT match archived E5 training source. "
        "Do not publish an alleged historical reproduction with changed data. "
        f"observed={sha256(random_data_path)} expected={EXPECTED_RANDOM_SHA}")
print(f"PASS: historical Random100k exact pair SHA={sha256(random_data_path)}")
print(f"Dataset location: {random_data_path}",flush=True)

## Task 3 — Run Round9 Random E1, E2 and the selected E5

The first two one-epoch continuations have **separate freshly initialized AdamW/cosine schedulers**. The last stage starts from Random E2, preserves optimizer/scheduler between E3, E4 and E5, and uses CosineAnnealingLR(T_max=4 × 1,562) even though it **stops after E5**. This precisely documents why we selected the E5 snapshot instead of the final E6.

The archived E5 checkpoint (if present) is **never used as initialization for training**; it can be read after the run only for parity diagnostics. The notebook's final training weights always derive from the provided SFT checkpoint and prior notebook stages.

In [ ]:
# STEP 8 — Exact Random E1 -> E2 -> E3/E4/E5 staged continuation.
STAGE5_TO_7=[
    {"id":"round9_random_e1", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round9_random_e2_restart", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round9_random_e3_e5_from_e2", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":3,"schedule_epochs":4}
]
assert [s["actual_epochs"] for s in STAGE5_TO_7]==[1,1,3]
assert [s["schedule_epochs"] for s in STAGE5_TO_7]==[1,1,4]

parent=ROUND7_PARENT
for spec in STAGE5_TO_7:
    parent=train_stage(spec,parent)
    stage_outputs[spec["id"]]=parent
    print(f"CHAIN {spec['id']} SHA256={sha256(parent)[:20]}",flush=True)

REPRODUCED_E5=stage_outputs["round9_random_e3_e5_from_e2"]
print(f"FINAL E5 REPLAY CHECKPOINT = {REPRODUCED_E5}",flush=True)